In [1]:
import pandas as pd

file_path = "02_Kommerzieller_Au_enhandel_201901010000_202601010000_Monat.csv"

# ---------- LOAD ----------
df = pd.read_csv(
    file_path,
    sep=";",
    encoding="utf-8-sig",
    decimal=",",
    thousands=".",
    na_values=["-"],
    dtype={"Datum von": str, "Datum bis": str}
)

# ---------- CLEAN COLUMN NAMES ----------
df.columns = (
    df.columns
    .str.replace(r"\[MWh\]", "", regex=True)
    .str.replace("Berechnete Auflösungen", "", regex=False)
    .str.strip()
)

# ---------- FIX DATE ----------
df["Datum von"] = pd.to_datetime(df["Datum von"], format="%d.%m.%Y")
df["Datum bis"] = pd.to_datetime(df["Datum bis"], format="%d.%m.%Y")

# ---------- FILL MISSING NETTOEXPORT ----------
# Für Monate mit fehlendem Nettoexport wird die monatliche Bilanz berechnet.
# Da Importwerte im Datensatz negativ sind, ergibt Export + Import den Nettoexport.

net_col = "Nettoexport"
export_cols = [col for col in df.columns if "(Export)" in col]
import_cols = [col for col in df.columns if "(Import)" in col]

missing_mask = df[net_col].isna()

df.loc[missing_mask, net_col] = (
    df.loc[missing_mask, export_cols].sum(axis=1)
    + df.loc[missing_mask, import_cols].sum(axis=1)
)

# ---------- RENAME DATE COLUMNS ----------
df = df.rename(columns={
    "Datum von": "datum_von",
    "Datum bis": "datum_bis"
})

# ---------- MELT WIDE → LONG ----------
df_long = df.melt(
    id_vars=["datum_von", "datum_bis"],
    var_name="handelsspalte",
    value_name="handelsmenge_mwh"
)

# ---------- SPLIT LAND / HANDELSART ----------
df_long["land"] = df_long["handelsspalte"].str.extract(r"^(.*?) \(")
df_long["handelsart"] = df_long["handelsspalte"].str.extract(r"\((.*?)\)")

# Nettoexport separat behandeln
df_long.loc[df_long["handelsspalte"] == "Nettoexport", "land"] = "Gesamt"
df_long.loc[df_long["handelsspalte"] == "Nettoexport", "handelsart"] = "Nettoexport"

# ---------- DROP TECHNICAL COLUMN ----------
df_long = df_long.drop(columns=["handelsspalte"])

# ---------- REMOVE EMPTY VALUES ----------
df_long = df_long.dropna(subset=["handelsmenge_mwh"])

# ---------- CLEAN COUNTRY NAMES ----------
df_long["land"] = (
    df_long["land"]
    .str.strip()
    .str.lower()
    .str.replace("ä", "ae")
    .str.replace("ö", "oe")
    .str.replace("ü", "ue")
    .str.replace("ß", "ss")
    .str.capitalize()
)

# Gesamt wieder korrekt groß schreiben
df_long.loc[df_long["land"] == "Gesamt", "land"] = "Gesamt"

# ---------- ADD ADDITIONAL COLUMNS ----------
df_long["handelsmenge_abs_mwh"] = df_long["handelsmenge_mwh"].abs()
df_long["ist_gesamt"] = df_long["land"] == "Gesamt"
df_long["jahr"] = df_long["datum_von"].dt.year
df_long["monat"] = df_long["datum_von"].dt.month

# ---------- SORT ----------
df_long = df_long.sort_values(["datum_von", "land", "handelsart"])

display(df_long.head())
display(df_long.tail())

,datum_von,datum_bis,handelsmenge_mwh,land,handelsart,handelsmenge_abs_mwh,ist_gesamt,jahr,monat
420,2019-01-01,2019-02-01,760895.0,Daenemark,Export,760895.0,False,2019,1
504,2019-01-01,2019-02-01,-499622.0,Daenemark,Import,499622.0,False,2019,1
1260,2019-01-01,2019-02-01,2574351.0,Frankreich,Export,2574351.0,False,2019,1
1344,2019-01-01,2019-02-01,-64918.0,Frankreich,Import,64918.0,False,2019,1
0,2019-01-01,2019-02-01,7423495.0,Gesamt,Nettoexport,7423495.0,True,2019,1


,datum_von,datum_bis,handelsmenge_mwh,land,handelsart,handelsmenge_abs_mwh,ist_gesamt,jahr,monat
1091,2025-12-01,2026-01-01,-138780.50,Schweden,Import,138780.50,False,2025,12
335,2025-12-01,2026-01-01,630941.37,Schweiz,Export,630941.37,False,2025,12
419,2025-12-01,2026-01-01,-161586.73,Schweiz,Import,161586.73,False,2025,12
671,2025-12-01,2026-01-01,931347.93,Tschechien,Export,931347.93,False,2025,12
755,2025-12-01,2026-01-01,-97814.67,Tschechien,Import,97814.67,False,2025,12


In [2]:
df_long = df_long[
    [
        "datum_von",
        "datum_bis",
        "jahr",
        "monat",
        "land",
        "handelsart",
        "handelsmenge_mwh",
        "handelsmenge_abs_mwh",
        "ist_gesamt"
    ]
]

In [3]:
df_long.to_csv(
    "02_bereinigter_aussenhandel_LV.csv",
    index=False,
    encoding="utf-8-sig"
)